# Adaptive AI Tutor

An AI tutor that **adapts its explanations to the learner's level** (e.g. beginner, intermediate, advanced) and runs behind a simple web interface.

**What it demonstrates**
- Level-aware prompting so the same question is answered differently per audience
- Building an interactive UI with Gradio
- Turning an LLM call into a usable web app

**Stack:** Python · OpenAI API · Gradio


In [1]:
!pip install gradio

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown


In [3]:
load_dotenv()  # Load environment variables from .env file

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key = openai_api_key)  

print("OpenAI API key loaded successfully.")

OpenAI API key loaded successfully.


In [4]:
from IPython.display import display, Markdown

def print_markdown(text):
    """Display text as markdown in jupyter"""
    display(Markdown(text))

In [5]:
def get_ai_tutor_response(user_question):
    """
    Sends a question to the OpenAI API, asking it to respond as an AI Tutor.

    Args:
        user_question (str): The question asked by the user.

    Returns:
        str: The AI's response, or an error message.
    """

    system_prompt = "You are a helpful and patient AI tutor. Explain concepts clearly and consisely."

    try:
        response = openai_client.chat.completions.create(
            model = "gpt-4o-mini",
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_question}
            ],
            temperature = 0.7,
        )
        ai_responce = response.choices[0].message.content
        return ai_responce

    except Exception as e:
        print(f"An error occurred: {e}")
        return "Sorry, I encountered and error while trying to respond. Please try again later."

In [6]:
test_question = "Can you explain python and how important is it for programming?"

print_markdown(f"Asking the AI Tutor: '{test_question}'")

tutor_answer = get_ai_tutor_response(test_question)
print_markdown(f"\n AI Tutor's response: \n")
print_markdown(tutor_answer)

Asking the AI Tutor: 'Can you explain python and how important is it for programming?'


 AI Tutor's response: 


Certainly! Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Here are some key features and concepts:

### Key Features of Python:

1. **Readability**: Python's syntax is designed to be clear and easy to understand, making it an excellent choice for beginners.

2. **Versatile**: Python can be used for various applications, including web development, data analysis, artificial intelligence, scientific computing, automation, and more.

3. **Large Standard Library**: Python comes with a vast standard library that supports many common programming tasks, such as file handling, web scraping, and data manipulation.

4. **Community Support**: Python has a large and active community, which means plenty of resources, libraries, and frameworks are available to help developers.

5. **Cross-Platform**: Python runs on various operating systems, including Windows, macOS, and Linux, making it highly versatile.

### Importance of Python in Programming:

1. **Ease of Learning**: Python is often recommended for beginners due to its straightforward syntax and semantics. This makes it easier to learn programming concepts.

2. **Wide Adoption**: Many organizations, from startups to tech giants, use Python for various applications. Learning Python can open up job opportunities in fields like software development, data science, machine learning, and more.

3. **Strong Community and Ecosystem**: The extensive libraries (like NumPy, Pandas, and TensorFlow) and frameworks (like Django and Flask) built on Python enhance its capabilities and simplify development processes.

4. **Rapid Development**: Python's simplicity allows for faster development cycles, which is particularly beneficial for startups and projects with tight deadlines.

5. **Interdisciplinary Use**: Python is used across many domains, including web development, data science, artificial intelligence, automation, and scientific research, making it a versatile skill to have.

### Conclusion:

In summary, Python is a powerful and accessible programming language that plays a significant role in the programming landscape. Its ease of use, versatility, and strong community support make it an essential language for both beginners and experienced developers alike. Whether you want to build web applications, analyze data, or create automation scripts, Python is a valuable tool in your programming toolkit.

In [7]:
import gradio as gr

c:\Users\princ\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
!pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 2.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ----------------------- ---------------- 1.3/2.2 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 7.2 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]



In [13]:
simple_ai_tutor_interface = gr.Interface(
    fn = get_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder = "Ask the AI tutor anything", label = "Your Question"),
    outputs = gr.Textbox(label = "AI Tutor's response"),
    title = "🤖 Simple AI Tutor",
    description = "Enter your question below and the AI Tutor will provide an explanation. Powered by OpenAI.",
    flagging_mode = "never"
)

print("Launching the Gradio...")
simple_ai_tutor_interface.launch()

Launching the Gradio...
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [14]:
def stream_ai_tutor_response(user_question):
    """
    Sends a question to the OpenAI API and streams the response as a generator.

    Args:
        user_question (str): The question asked by the user.

    Yields:
        str: Chunks of the AI's response.
    """

    system_prompt = "You are a helpful and patient AI tutor. Explain concepts clearly and consisely."

    try:
        response = openai_client.chat.completions.create(
            model = "gpt-4o-mini",
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_question}
            ],
            temperature = 0.7,
            stream = True
        )
        full_response ="" 

        for chunk in response:
            if chunk.choices[0].delta and chunk.choices[0].delta.content:
                text_chunk = chunk.choices[0].delta.content
                full_response += text_chunk
                yield full_response

    except Exception as e:
        print(f"An error occurred: {e}")
        return "Sorry, I encountered and error while trying to respond. Please try again later."

In [15]:
streaming_ai_tutor_interface = gr.Interface(
    fn = stream_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder = "Ask the AI tutor anything", label = "Your Question"),
    outputs = gr.Markdown(label = "AI Tutor's streaming response", container = True, height = 300),
    title = "🤖 AI Tutor with Streaming",
    description = "Enter your question below and the AI Tutor will provide an explanation. Powered by OpenAI.",
    flagging_mode = "never"
)

print("Launching Streaming Gradio Interface...")
streaming_ai_tutor_interface.launch()

Launching Streaming Gradio Interface...
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [23]:
explanation_levels = {
    1: "like I'm 5 years old",
    2: "like I'm 10 years old",
    3: "like a high school student",
    4: "like a college student",
    5: "like an expert in the field",
}

In [24]:
def stream_ai_tutor_response_with_level(user_question, explanation_level_value):
    """
    Streams AI Tutor response based on user question and selected explanation level.

    Args:
        user_question (str): The question from the user.
        explanation_level_value (int): The value from the slider (1-5).

    Yields:
        str: Chunks of the AI's response.
    """

    level_description = explanation_levels.get(explanation_level_value, "Clearly and concisely")

    system_prompt = f"You are a helpful and patient AI tutor. Explain the following concepts {level_description}."

    print(f"DEBUG: Using System Prompt: '{system_prompt}'") 

    try:
        response = openai_client.chat.completions.create(
            model = "gpt-4o-mini",
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_question}
            ],
            temperature = 0.7,
            stream = True
        )
        full_response ="" 

        for chunk in response:
            if chunk.choices[0].delta and chunk.choices[0].delta.content:
                text_chunk = chunk.choices[0].delta.content
                full_response += text_chunk
                yield full_response

    except Exception as e:
        print(f"An error occurred: {e}")
        return "Sorry, I encountered an error while trying to respond. Please try again later."

In [ ]:
streaming_ai_tutor_interface_with_level = gr.Interface(
    fn = stream_ai_tutor_response_with_level,
    inputs = [
        gr.Textbox(lines = 2, placeholder = "Ask the AI tutor a question...", label = "Your Question"),
        gr.Slider(
            minimum = 1,
            maximum = 5,
            step = 1,
            value = 3,
            label = "Explanation Level",
        )
        ],
    outputs = gr.Markdown(label = "AI Tutor's streaming response", container = True, height = 300),
    title = "🤖 AI Tutor with Streaming",
    description = "Enter your question below and the AI Tutor will provide an explanation. Powered by OpenAI.",
    flagging_mode = "never"
)

print("Launching Streaming Gradio Interface with slider...")
streaming_ai_tutor_interface_with_level.launch()

Launching Streaming Gradio Interface with slider...
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


DEBUG: Using System Prompt: 'You are a helpful and patient AI tutor. Explain the following concepts like I'm 5 years old.'
DEBUG: Using System Prompt: 'You are a helpful and patient AI tutor. Explain the following concepts like I'm 10 years old.'
DEBUG: Using System Prompt: 'You are a helpful and patient AI tutor. Explain the following concepts like a high school student.'
DEBUG: Using System Prompt: 'You are a helpful and patient AI tutor. Explain the following concepts like an expert in the field.'
